# 02 — Generate and Inspect Gully Labels

This notebook covers:
1. Synthetic label generation from MOLA slope/aspect thresholds
2. Mask overlay visualisation on HiRISE
3. Class balance analysis (positive pixel fraction per patch)
4. Patch augmentation preview


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from pathlib import Path

from scripts.utils import load_config
cfg = load_config('../config.yaml')
print('Config OK.')

In [ ]:
# ── Synthetic label preview from MOLA slope ──────────────────────────────────
from scripts.features.morphometric import compute_slope, compute_aspect

dem_dir = Path('../data/raw/mola')
dem_files = sorted(dem_dir.glob('*.tif'))

if dem_files:
    with rasterio.open(dem_files[0]) as src:
        dem = src.read(1).astype(np.float32)
    
    slope = compute_slope(dem, resolution=463.0)
    aspect = compute_aspect(dem, resolution=463.0)
    
    # Synthetic gully mask: slope > 15° and poleward aspect
    synthetic_mask = ((slope > 15) & (aspect > 135) & (aspect < 315)).astype(np.uint8)
    
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(dem, cmap='terrain'); axes[0].set_title('DEM (MOLA)')
    axes[1].imshow(slope, cmap='YlOrRd', vmax=45); axes[1].set_title('Slope (degrees)')
    axes[2].imshow(aspect, cmap='hsv'); axes[2].set_title('Aspect (degrees)')
    axes[3].imshow(synthetic_mask, cmap='binary'); axes[3].set_title('Synthetic Gully Mask')
    
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.savefig('../data/outputs/reports/synthetic_labels_preview.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Positive pixel fraction: {synthetic_mask.mean():.3f}')
else:
    print('No MOLA DEM found. Run: python main.py --step download')

In [ ]:
# ── Patch class-balance analysis ─────────────────────────────────────────────
label_dir = Path('../data/processed/labels')
label_files = sorted(label_dir.glob('*.tif'))

if label_files:
    fractions = []
    for lf in label_files:
        with rasterio.open(lf) as src:
            mask = src.read(1).astype(np.float32)
        fractions.append(float(mask.mean()))
    
    fractions = np.array(fractions)
    print(f'Label files: {len(fractions)}')
    print(f'Positive fraction — mean={fractions.mean():.4f}  '
          f'std={fractions.std():.4f}  '
          f'max={fractions.max():.4f}  '
          f'min={fractions.min():.4f}')
    
    plt.figure(figsize=(8, 4))
    plt.hist(fractions, bins=40, color='#e74c3c', alpha=0.75, edgecolor='white')
    plt.axvline(fractions.mean(), color='black', linestyle='--', label=f'Mean={fractions.mean():.3f}')
    plt.xlabel('Positive pixel fraction')
    plt.ylabel('Count')
    plt.title('Gully Mask Class Balance (per label file)')
    plt.legend()
    plt.tight_layout()
    plt.savefig('../data/outputs/reports/class_balance.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No label files. Run: python main.py --step labels')

In [ ]:
# ── Patch augmentation preview ───────────────────────────────────────────────
patch_dir = Path('../data/processed/patches')
patch_files = sorted(patch_dir.glob('image_*.npy'))[:6]
mask_files  = sorted(patch_dir.glob('mask_*.npy'))[:6]

if patch_files and mask_files:
    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    for i, (pf, mf) in enumerate(zip(patch_files, mask_files)):
        img = np.load(pf)        # (C, H, W)
        mask = np.load(mf)       # (H, W) or (1, H, W)
        if mask.ndim == 3: mask = mask[0]
        
        # Display first 3 channels as RGB
        rgb = np.clip(img[:3].transpose(1, 2, 0), 0, 1) if img.shape[0] >= 3 else img[0]
        axes[0, i].imshow(rgb if img.shape[0] >= 3 else rgb, cmap='gray')
        axes[0, i].set_title(f'Image {i+1}')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(mask, cmap='binary')
        axes[1, i].set_title(f'Mask {i+1} ({mask.mean():.3f})')
        axes[1, i].axis('off')
    
    plt.suptitle('Training Patches (image top, mask bottom)', fontsize=13)
    plt.tight_layout()
    plt.savefig('../data/outputs/reports/patch_preview.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No patches found. Run: python main.py --step labels')